In [ ]:
import json

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import tifffile as tiff
from nibabel.affines import apply_affine
from PIL import Image

In [ ]:
img_id = "3196"

hr_path = (
    f"data/raw/high_res/B20_{img_id}.tif"
)
hr_affine_path = f"data/raw/high_res/B20_{img_id}_affine.json"

lr_path = f"data/raw/low_res/pm{img_id}o.mnc"

In [ ]:
# 1. Load HR image and affine
with tiff.TiffFile(hr_path) as tif:
    high_res_image_page = tif.pages[0]

hr_data = high_res_image_page.asarray()

hr_affine = np.array(json.load(open(hr_affine_path)))

# 2. Load LR image and affine
lr_nib = nib.load(lr_path)
lr_data = np.array(lr_nib.dataobj)
lr_affine = lr_nib.affine
lr_affine_inv = np.linalg.inv(lr_affine)

In [ ]:
# Visualize the two images to check they are correctly loaded

hr_resized = Image.fromarray(hr_data).resize(
    (hr_data.shape[::-1] / np.array([64])).astype(int), resample=Image.BILINEAR
)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(hr_resized, cmap="gray")
plt.title("High-Resolution Image")
plt.subplot(1, 2, 2)
plt.imshow(lr_data[:, 0, :], cmap="gray")
plt.title("Low-Resolution Image")
plt.show()

In [ ]:
### Le due immagini sono orientate in modo diverso.
### Il "vero" in basso a sinistra, in HR e' a (0, max) mentre in LR e' a (0, 0)
### Questo vuol dire che concettualmente la HR avra' l'asse y invertito rispetto la LR
### Il fatto e' che plt applica gia' questa inversione, quindi visualizzandole cosi, la HR sembrera' quella orientata correttamente

In [ ]:
## MAPPING TEST

# 3. Point on HR image (z,x in HR space)
hr_point = np.array([70000, 97000])


# Plot the point on the resized HR image
hr_point_resized = hr_point / 64
plt.imshow(hr_resized, cmap="gray")
plt.scatter(hr_point_resized[1], hr_point_resized[0], c="red", marker="x")
plt.title("HR Image with Crop Point")
plt.show()

In [ ]:
def OLD_map_to_lr_space(hr_adj_coord, img_shape_z):
    # ISSUE: why i need to flip BOTH axis based on max_z HR value?? 
    # then go HR -> world -> LR
    hr_coord = np.array([img_shape_z - hr_adj_coord[0], 0, img_shape_z - hr_adj_coord[2]])
    global_coord = apply_affine(hr_affine, hr_coord)
    lr_coord = apply_affine(lr_affine_inv, global_coord).astype(int)
    return lr_coord

def map_to_lr_space(hr_adj_coord):
    # go HR -> world -> LR

    # swap the axis to get a point in (x,z) in Hr space
    hr_coord = np.array([hr_adj_coord[2], 0, hr_adj_coord[0]])

    global_coord = apply_affine(hr_affine, hr_coord)
    lr_coord = apply_affine(lr_affine_inv, global_coord).astype(int)

    # since the two affines swaps the axes, the resulting lr_coord is in (z,x)

    return lr_coord


hr_point_3 = np.array([hr_point[0], 0, hr_point[1]]) # (z, y, x) in HR space
lr_point = map_to_lr_space(hr_point_3) # (z, y, x) in LR space

print(lr_point)

# Plot the point on the LR image
plt.imshow(lr_data[:, 0, :], cmap="gray")
plt.scatter(lr_point[2], lr_point[0], c="red", marker="x")
plt.title("LR Image with Mapped Point")
plt.show()

In [ ]:
### TRY TO MAP A BOUNDING BOX INSTEAD OF A POINT, TO CHECK IF THE ISSUE IS THE SAME

In [ ]:
hr_point_A = np.array([66000, 93000])
hr_point_B = np.array([73000, 102000])

# plot this box on the HR image
hr_point_A_resized = hr_point_A / 64
hr_point_B_resized = hr_point_B / 64
plt.imshow(hr_resized, cmap="gray")

# Plot a rectangle between the two points
plt.plot(
    [hr_point_A_resized[1], hr_point_B_resized[1], hr_point_B_resized[1], hr_point_A_resized[1], hr_point_A_resized[1]],
    [hr_point_A_resized[0], hr_point_A_resized[0], hr_point_B_resized[0], hr_point_B_resized[0], hr_point_A_resized[0]],
    c="red",
)
plt.title("HR Image with Crop Box")
plt.show()

In [ ]:
# 3. Map HR voxel coordinates to LR voxel coordinates
hr_point_A_3 = np.array([hr_point_A[0], 0, hr_point_A[1]]) # (z, y, x) in HR space
hr_point_B_3 = np.array([hr_point_B[0], 0, hr_point_B[1]]) # (z, y, x) in HR space

lr_point_A = map_to_lr_space(hr_point_A_3)
lr_point_B = map_to_lr_space(hr_point_B_3)

print("LR Point A:", lr_point_A)
print("LR Point B:", lr_point_B)

# Plot the box on the LR image
plt.imshow(lr_data[:, 0, :], cmap="gray")
plt.plot(
    [lr_point_A[2], lr_point_B[2], lr_point_B[2], lr_point_A[2], lr_point_A[2]],
    [lr_point_A[0], lr_point_A[0], lr_point_B[0], lr_point_B[0], lr_point_A[0]],
    c="red",
)
plt.title("LR Image with Mapped Box")
plt.show()

In [ ]:
hr_point_A = np.array([66000, 93000])
hr_point_B = np.array([73000, 102000])

# z y x
LR Point A: [2591 -305 4393]
LR Point B: [2261 -305 4818]


# x z
66000 93000 => 4393 2591 # top left
73000 102000 => 4818 2261 # bottom right